# Tome C3, Chapter 11: Randomness, Interaction and Proof (Python companion)

Exercises first, solutions at the end. We amplify a weak coin, cover a big set of random strings with a few shifts as Lautemann did, hunt for Adleman's single good random string, play the graph non-isomorphism game against honest and lying provers, run the sum-check protocol on real formulas and catch a cheater, verify that a zero-knowledge simulator produces exactly the real transcripts, see why a naive proof checker needs the PCP theorem, and finally extract clean random bits from dirty sources.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(11)
rng = np.random.default_rng(11)

## Exercises

**P11.1 (★) Amplification.** A machine answers correctly with probability $\frac12+\frac1q$. Compute exactly the error of the majority of $k$ independent runs, for $q=4$ and $q=10$, and compare with the Hoeffding bound $e^{-2k/q^2}$ used in Exercise 11.2. How many runs bring the error below $10^{-6}$?

**P11.2 (★★) Lautemann's shifts.** In $\{0,1\}^m$ with $m=12$, take a random set $A$ containing a fraction $1-\delta$ of the points. How many random shifts $s_1,\dots,s_k$ are needed, typically, for the translates $A\oplus s_i$ to cover everything? Compare with the counting in the proof of Theorem 11.4, and check that for a set of density $\delta$ the same number of translates covers at most $k\delta 2^m$ points.

**P11.3 (★★) Adleman's good string.** Freivalds' test checks $AB=C$ for $2\times2$ matrices over $\mathbb F_2$ by picking a random vector $r$ and comparing $A(Br)$ with $Cr$. With $k$ independent vectors, find the fraction of random strings $(r_1,\dots,r_k)$ that give the right answer on *all* $2^{12}$ inputs $(A,B,C)$, and compare with the union bound of Theorem 11.5.

**P11.4 (★★) Graph non-isomorphism.** Implement the protocol of Proposition 11.6 for graphs on $6$ vertices: the verifier picks $b$ and a random permutation, sends a relabelled $G_b$, and the prover must say which graph it came from. Measure the acceptance probability of an honest prover when $G_1\not\cong G_2$, and of the best possible prover when $G_1\cong G_2$, for one and for two rounds.

**P11.5 (★★★) Sum-check on a formula.** Arithmetise a random 3-CNF formula $\varphi$ with $v$ variables over $\mathbb F_p$ and run the sum-check protocol on the claim "$\varphi$ has $H$ models". Check that the honest prover always convinces the verifier, then let a prover lie about $H$ and fight as well as it can (sending, in each round where its claim is still false, a polynomial that passes the check and agrees with the true one at $d$ points of its choice). How often does it escape, for small primes $p$, compared with the bound $vd/p$ of Theorem 11.7?

**P11.6 (★★) A simulator at work.** For the zero-knowledge protocol for graph isomorphism (Theorem 11.11), compute exactly the distribution of the transcripts $(H,b,\sigma)$ of a real interaction and of the simulator's output, for an honest verifier and for a verifier that picks $b$ as a function of $H$. Count the rewinds the simulator needs.

**P11.7 (★★) Why naive proofs are not checkable.** A naive proof that a graph is $3$-colourable is a colouring, and a naive verifier checks one random edge. Build graphs that are not $3$-colourable but where some colouring violates a single edge, and compute the probability that the naive verifier catches the lie. Compare with the constant rejection probability promised by the PCP theorem (Corollary 11.14).

**P11.8 (★★) Extracting randomness.** (a) Run von Neumann's extractor (Exercise 11.10) on biased coins, check the uniformity of the output and its rate $p(1-p)$ per toss. (b) For flat sources $X$ uniform on a random set of $2^k$ strings of length $n=12$, and the universal family $h(x)=Mx+c$ over $\mathbb F_2$, compute exactly the statistical distance $\Delta\bigl((h,h(X)),(h,U_m)\bigr)$ and compare with the bound $\frac12\sqrt{2^{m-k}}$ of Theorem 11.16.

## Solutions

### P11.1
The number of correct runs is binomial; the majority errs when at most half the runs are correct (we take $k$ odd).

In [ ]:
def majority_error(q, k):
    p = 0.5 + 1 / q
    return sum(math.comb(k, j) * p**j * (1 - p)**(k - j) for j in range(0, k // 2 + 1))

fig, ax = plt.subplots(figsize=(6, 3.6))
for q in [4, 10]:
    ks = list(range(1, 8 * q * q, 2))
    ax.semilogy(ks, [majority_error(q, k) for k in ks], label=f"exact, q = {q}")
    ax.semilogy(ks, [math.exp(-2 * k / q**2) for k in ks], "--", label=f"Hoeffding, q = {q}")
    k = 1
    while majority_error(q, k) > 1e-6:
        k += 2
    kb = math.ceil(q * q / 2 * math.log(1e6))
    print(f"q = {q:2d}: error below 1e-6 from k = {k} runs (Hoeffding guarantees it from k = {kb})")
ax.set_xlabel("number of runs k"); ax.set_ylabel("error of the majority"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The error falls exponentially in $k/q^2$, as the proof of Proposition 11.2 says: the number of runs needed grows like $q^2\log\frac1\varepsilon$. Hoeffding's bound is pessimistic by a constant factor in the exponent only; the true error decays at rate about $2k/q^2$ too, with a polynomial prefactor that the bound throws away. A bias of $\frac1{10}$ costs a hundred times more runs than a bias of $1$ would, but never an exponential.

### P11.2
Points of $\{0,1\}^{12}$ are integers below $4096$, and a shift is an exclusive or.

In [ ]:
m = 12; N = 1 << m
def shifts_needed(A_mask, rng_py, kmax=40):
    covered = np.zeros(N, dtype=bool); pts = np.arange(N); Aidx = pts[A_mask]
    for k in range(1, kmax + 1):
        s = rng_py.randrange(N)
        covered[Aidx ^ s] = True
        if covered.all():
            return k
    return None

rp = random.Random(3)
for delta in [0.5, 0.2, 0.05, 2**-6]:
    ks = []
    for t in range(30):
        A = rng.random(N) >= delta
        ks.append(shifts_needed(A, rp))
    bound = math.floor(m / math.log2(1 / delta)) + 1     # k with delta^k 2^m < 1
    print(f"density 1-delta, delta = {delta:.4f}: shifts needed  mean {np.mean(ks):5.2f}, max {max(ks):2d};"
          f"  delta^k 2^m < 1 from k = {bound}")
# the small side: a set of density delta covers at most k*delta*2^m points
delta, k = 2**-6, 3
A = rng.random(N) < delta
cover = set()
for s in rp.sample(range(N), k):
    cover |= {a ^ s for a in np.flatnonzero(A)}
print(f"small set: |A| = {A.sum()}, {k} translates cover {len(cover)} points <= {k * A.sum()} of {N}")

The proof of Theorem 11.4 fixes a point $z$: it escapes all $k$ random translates with probability $\delta^k$, so a union bound over the $2^m$ points gives a covering as soon as $\delta^k2^m<1$. The experiment matches this counting closely: the typical number of shifts is about $\log_2 2^m/\log_2\frac1\delta$, plus a little. On the other side the count is trivial and exact: $k$ translates of a small set cannot cover more than $k|A|$ points. Lautemann's $\exists\forall$ formula is precisely "there are shifts such that every point is covered".

### P11.3
A $2\times2$ matrix over $\mathbb F_2$ is $4$ bits, so an input $(A,B,C)$ is $12$ bits and a random vector $r$ is $2$ bits.

In [ ]:
mats = [np.array(b, dtype=np.int64).reshape(2, 2) for b in itertools.product((0, 1), repeat=4)]
vecs = [np.array(v, dtype=np.int64) for v in itertools.product((0, 1), repeat=2)]
# for each input, the set of vectors r that expose AB != C (empty if AB = C)
witness_sets = Counter()
for A in mats:
    for B in mats:
        AB = A @ B % 2
        for Cm in mats:
            D = (AB - Cm) % 2
            bad = frozenset(i for i, r in enumerate(vecs) if not (D @ r % 2).any())  # r misses the error
            witness_sets[(D.any(), bad)] += 1
wrong = sum(c for (isdiff, bad), c in witness_sets.items() if isdiff)
print("inputs with AB != C:", wrong, "of", 2**12)
worst = max(len(bad) for (isdiff, bad) in witness_sets if isdiff) / 4
print("worst error of one vector on a wrong input:", worst)
for k in range(1, 6):
    good = 0
    for rs in itertools.product(range(4), repeat=k):
        if all(not set(rs) <= bad for (isdiff, bad) in witness_sets if isdiff):
            good += 1
    union = min(1.0, wrong * worst**k)
    print(f"k = {k}: good strings {good:5d} of {4**k:5d} = {good / 4**k:.3f};  union bound promises >= {1 - union:.3f}")

Freivalds' test errs with probability at most $\frac12$ on a wrong input (the vector $r$ must lie in the kernel of $AB-C\ne0$, a proper subspace). The union bound of Theorem 11.5 needs $2^{12}2^{-k}<1$, so $k\ge13$, before it promises a single good string. Reality is far kinder: from $k=2$ on, some strings are already good for all inputs, namely the pairs of vectors that span $\mathbb F_2^2$ (with $r_1=e_1$, $r_2=e_2$ the test computes $AB$ and $C$ entirely). This is the general lesson of Adleman's theorem: a good random string exists, and the union bound is a lazy way of proving it; finding one explicitly is derandomisation.

### P11.4
Graphs on $6$ vertices are sets of edges; we can afford the $720$ permutations to test isomorphism by brute force.

In [ ]:
n = 6
PERMS = list(itertools.permutations(range(n)))
def relabel(E, pi):
    return frozenset(frozenset((pi[a], pi[b])) for a, b in (tuple(e) for e in E))
def canon(E):
    return min(tuple(sorted(tuple(sorted(e)) for e in relabel(E, pi))) for pi in PERMS)
def random_graph(pe=0.5):
    return frozenset(frozenset(e) for e in itertools.combinations(range(n), 2) if random.random() < pe)

def run(G1, G2, prover, rounds, trials=400):
    ok = 0
    for _ in range(trials):
        acc = True
        for _ in range(rounds):
            b = random.randrange(2)
            H = relabel((G1, G2)[b], random.choice(PERMS))
            acc &= prover(G1, G2, H) == b
        ok += acc
    return ok / trials

def honest(G1, G2, H):        # unlimited power: test isomorphism
    return 0 if canon(H) == canon(G1) else 1
def best_liar(G1, G2, H):     # G1 = G2 up to relabelling: H carries no information
    return random.randrange(2)

G1 = random_graph()
while True:
    G2 = random_graph()
    if canon(G2) != canon(G1):
        break
G1b = relabel(G1, random.choice(PERMS))
t0 = time.perf_counter()
for rounds in [1, 2]:
    print(f"{rounds} round(s): honest prover, G1 not~ G2: {run(G1, G2, honest, rounds):.3f};"
          f"   any prover, G1 ~ G2: {run(G1, G1b, best_liar, rounds):.3f}")
print(f"({time.perf_counter() - t0:.1f} s)")

When the graphs are not isomorphic, the honest prover sees which class $H$ belongs to and is never wrong: completeness $1$. When they are isomorphic, a uniformly relabelled $G_1$ and a uniformly relabelled $G_2$ have *the same distribution*, so whatever the prover computes it guesses $b$ with probability exactly $\frac12$ per round, and $\frac14$ for two rounds. The prover's unlimited power is useless against a coin it cannot see; this is the whole idea of private coins.

### P11.5
We code a clause as three literals $\pm i$; its arithmetisation is $1-\prod(1-\ell)$, with $x$ for a positive and $1-x$ for a negative literal. The degree in each variable is at most the number of clauses containing it, and the prover sends each $g_i$ by its values at $0,1,\dots,d$.

In [ ]:
def random_3cnf(v, mcl, rr):
    return [tuple(rr.choice((-1, 1)) * x for x in rr.sample(range(1, v + 1), 3)) for _ in range(mcl)]
def g_eval(F, pt, p):        # pt: list of field elements, pt[i-1] = value of x_i
    val = 1
    for cl in F:
        miss = 1
        for l in cl:
            x = pt[abs(l) - 1]
            miss = miss * ((1 - x) if l > 0 else x) % p
        val = val * (1 - miss) % p
    return val
def partial_sum(F, prefix, p, v):     # sum over Boolean completions of the prefix
    rest = v - len(prefix)
    return sum(g_eval(F, list(prefix) + list(b), p) for b in itertools.product((0, 1), repeat=rest)) % p
def lagrange(vals, x, p):            # polynomial through (j, vals[j]) evaluated at x
    tot = 0
    for j, yj in enumerate(vals):
        num = den = 1
        for i in range(len(vals)):
            if i != j:
                num = num * (x - i) % p; den = den * (j - i) % p
        tot = (tot + yj * num * pow(den, p - 2, p)) % p
    return tot

def sumcheck(F, v, d, H, p, rr, cheat=False):
    claim, prefix = H % p, []
    for i in range(v):
        vals = [partial_sum(F, prefix + [t], p, v) for t in range(d + 1)]   # true g_i at 0..d
        diff = (claim - vals[0] - vals[1]) % p
        if cheat and diff:   # still lying: send s_i + c (X - a_1)...(X - a_d), passing the check
            while True:
                a = rr.sample(range(p), d)
                q0, q1 = math.prod(-x for x in a) % p, math.prod(1 - x for x in a) % p
                if (q0 + q1) % p:
                    break
            c = diff * pow(q0 + q1, p - 2, p) % p
            vals = [(y + c * math.prod(t - x for x in a)) % p for t, y in enumerate(vals)]
        if (vals[0] + vals[1]) % p != claim:
            return False
        r = rr.randrange(p)
        claim = lagrange(vals, r, p); prefix.append(r)
    return g_eval(F, prefix, p) == claim

rr = random.Random(5)
v, mcl = 6, 10
F = random_3cnf(v, mcl, rr)
d = max(Counter(abs(l) for cl in F for l in cl).values())
H = sum(all(any((a[abs(l) - 1] == 1) == (l > 0) for l in cl) for cl in F) for a in itertools.product((0, 1), repeat=v))
print(f"v = {v}, clauses = {mcl}, degree per variable d = {d}, true number of models H = {H}")
t0 = time.perf_counter()
print("honest prover accepted in 20/20 runs:", all(sumcheck(F, v, d, H, 101, rr) for _ in range(20)))
for p in [37, 101, 401, 1601]:
    T = 1000
    esc = sum(sumcheck(F, v, d, H + 1, p, rr, cheat=True) for _ in range(T)) / T
    print(f"p = {p:3d}: liar claiming H+1 escapes {esc:.3f};  bound v d / p = {min(1, v * d / p):.3f}")
print(f"({time.perf_counter() - t0:.1f} s)")

The honest prover always passes. The liar passes every consistency check by construction, so it is caught only at the end, when the verifier evaluates $g$ itself; it escapes only if in some round the random $r_i$ hits one of the $d$ points where its polynomial agrees with the true one; from then on its claim is true and it can play honestly. This liar makes the most of the proof's weak spot, and its escape rate, about $1-(1-d/p)^v$, stays below the union bound $vd/p$ (up to sampling noise); for tiny primes it often wins, and its chances collapse as $p$ grows. In Corollary 11.8 the prime is of size about $2^n$, and the liar's chances are negligible. Notice the price the verifier pays: one evaluation of $g$ and $v$ small interpolations, while the prover summed over $2^v$ points.

### P11.6
We enumerate everything for a small graph: $G_1$ a path on $4$ vertices, $G_2=\tau(G_1)$. A transcript is $(H,b,\sigma)$ with $\sigma(G_b)=H$.

In [ ]:
n4 = 4
P4 = list(itertools.permutations(range(n4)))
def rel4(E, pi):
    return frozenset(frozenset((pi[a], pi[b])) for a, b in (tuple(e) for e in E))
def compose(a, b):      # (a o b)(i) = a[b[i]]
    return tuple(a[b[i]] for i in range(len(b)))
def inverse(a):
    inv = [0] * len(a)
    for i, ai in enumerate(a):
        inv[ai] = i
    return tuple(inv)
G1 = frozenset(frozenset(e) for e in [(0, 1), (1, 2), (2, 3)])
tau = (2, 0, 3, 1)
G2 = rel4(G1, tau)

def real(verifier):
    D = Counter()
    for pi in P4:                      # prover: H = pi(G1)
        H = rel4(G1, pi)
        for b, pb in verifier(H):      # verifier's choice, possibly depending on H
            sigma = pi if b == 0 else compose(pi, inverse(tau))   # sigma(G_b) = H
            D[(H, b, sigma)] += pb / len(P4)
    return D

def simulated(verifier):
    D = Counter(); rewinds = 0.0
    # simulator: guess c, pick rho, H = rho(G_c), ask the verifier; restart if b != c
    out = Counter()
    for c in (0, 1):
        for rho in P4:
            H = rel4((G1, G2)[c], rho)
            for b, pb in verifier(H):
                if b == c:
                    out[(H, b, rho)] += pb / (2 * len(P4))
    succ = sum(out.values())             # probability that one attempt succeeds
    for t, w in out.items():
        D[t] = w / succ                  # output distribution, conditioned on success
    return D, 1 / succ

honest_V = lambda H: [(0, 0.5), (1, 0.5)]
sneaky_V = lambda H: [(len({min(e) for e in H}) % 2, 1.0)]     # b depends on H
for name, V in [("honest verifier", honest_V), ("verifier choosing b from H", sneaky_V)]:
    R = real(V); S, attempts = simulated(V)
    dist = 0.5 * sum(abs(R[t] - S[t]) for t in set(R) | set(S))
    print(f"{name:28s}: {len(R)} transcripts, statistical distance real/simulated = {dist:.2e},"
          f" expected attempts {attempts:.2f}")

The two distributions coincide exactly, for both verifiers: the simulator's output is the real conversation, though the simulator never knew $\tau$. The reason is the one in the proof of Theorem 11.11: $H=\rho(G_c)$ is a uniformly random relabelling of $G_1$ whatever $c$ is, so the verifier's choice cannot depend on $c$, a guess succeeds with probability $\frac12$, and the simulator needs two attempts on average. A transcript that anyone can produce without the secret cannot teach anything about the secret.

### P11.7
Take $K_4$ (not $3$-colourable) and attach long chains of triangles, which are $3$-colourable and add many edges; a colouring that is proper except on one edge of the $K_4$ survives the naive verifier on all other edges.

In [ ]:
def chain_graph(t):
    E = [(a, b) for a, b in itertools.combinations(range(4), 2)]   # K4 on 0..3
    last = 3; nxt = 4
    for _ in range(t):                                            # triangles hanging off the K4
        E += [(last, nxt), (last, nxt + 1), (nxt, nxt + 1)]
        last = nxt + 1; nxt += 2
    return nxt, E

def min_violations(nv, E):
    # dynamic programming would do; for these sizes, fix the K4 colours and colour the chain greedily
    best = None
    for cols in itertools.product(range(3), repeat=4):
        col = list(cols) + [None] * (nv - 4)
        for a, b in E:                       # chain vertices: pick a colour avoiding neighbours
            for x in (a, b):
                if col[x] is None:
                    used = {col[y] for (u, w) in E for y in (u, w) if x in (u, w) and y != x and col[y] is not None}
                    col[x] = min(set(range(3)) - used)
        viol = sum(col[a] == col[b] for a, b in E)
        best = viol if best is None else min(best, viol)
    return best

for t in [0, 5, 20, 100]:
    nv, E = chain_graph(t)
    mv = min_violations(nv, E)
    print(f"{nv:4d} vertices, {len(E):4d} edges: best colouring violates {mv} edge(s);"
          f" naive verifier catches the lie with probability {mv / len(E):.4f}")

The graph is never $3$-colourable, yet the best cheating colouring is wrong on a single edge, so the naive verifier, reading two colours, catches it with probability $1/|E|$, which tends to $0$. To reject with a probability bounded below one would have to read a constant fraction of the proof. The PCP theorem says that this is an artefact of the naive encoding: there is a way to write proofs (much more redundant than a colouring) such that a false claim is wrong *everywhere*, and a constant number of random probes suffice. Corollary 11.14 is the same statement in the language of formulas: a gap of a constant fraction of violated clauses.

### P11.8
(a) Von Neumann's trick on coins of several biases.

In [ ]:
for p in [0.5, 0.8, 0.95]:
    tosses = rng.random(200_000) < p
    a, b = tosses[0::2], tosses[1::2]
    keep = a != b
    out = a[keep].astype(int)               # heads-tails -> 1 here; any fixed convention works
    pairs = Counter(zip(out[0::2], out[1::2]))
    print(f"p = {p:.2f}: {len(out):6d} bits from {len(tosses)} tosses (rate {len(out) / len(tosses):.4f},"
          f" theory p(1-p) = {p * (1 - p):.4f}); mean {out.mean():.3f};"
          f" pairs {[round(pairs[k] / sum(pairs.values()), 3) for k in sorted(pairs)]}")

The bits are fair and independent whatever the bias (the four pairs come out with frequencies near $\frac14$), but the rate collapses for a strongly biased coin. And von Neumann's trick needs independent tosses; for a general source of min-entropy $k$ a seed is necessary (Exercise 11.9).

(b) The leftover hash lemma, exactly. For a fixed $h$ we compute the distribution of $h(X)$ by counting; the distance of the pairs $(h,h(X))$ and $(h,U_m)$ is the average over $h$ of $\Delta(h(X),U_m)$, which we estimate on $200$ random functions $h$.

In [ ]:
n = 12
def lhl_distance(k, m, nh=200):
    S = rng.choice(1 << n, size=1 << k, replace=False)            # flat source of min-entropy k
    bits = ((S[:, None] >> np.arange(n)) & 1).astype(np.int64)    # |S| x n
    w = 1 << np.arange(m)
    tot = 0.0
    for _ in range(nh):
        M = rng.integers(0, 2, size=(n, m)); c = rng.integers(0, 2, size=m)
        y = ((bits @ M + c) % 2) @ w
        counts = np.bincount(y, minlength=1 << m) / len(S)
        tot += 0.5 * np.abs(counts - 1 / (1 << m)).sum()
    return tot / nh

t0 = time.perf_counter()
k = 8
for m in [2, 4, 6, 7, 8, 9]:
    print(f"k = {k}, m = {m}: distance {lhl_distance(k, m):.4f};  bound (1/2) sqrt(2^(m-k)) = {0.5 * math.sqrt(2**(m - k)):.4f}")
print(f"({time.perf_counter() - t0:.1f} s)")

The distance stays below the bound and follows its shape: each bit of output beyond what the min-entropy can pay for costs a factor $\sqrt2$, and once $m>k$ the output cannot be close to uniform at all, since $h(X)$ takes at most $2^k$ values. So from $k$ bits of min-entropy one extracts about $k-2\log_2\frac1\varepsilon$ bits that are $\varepsilon$-close to uniform, with a seed (the choice of $h$) that is itself reusable and public. This is the lemma behind privacy amplification in quantum key distribution and behind many proofs in Tome C4.